In [2]:
# --- Ek maskeleme v2: noktalama + kendi adın doğrudan ---

# --- 1. Orijinalden tekrar başla (v1'in çıktısını değil) ---
ornek = pd.read_csv("../data/external/salih_kurumsal_ornek.csv")
ornek["mesaj_eski"] = ornek["mesaj"]

# --- 2. Kendi adın: büyük/küçük harf ve Türkçe/İngilizce karakterlerin hepsi ---
# [İIi] → İ, I ya da i;  [ÜUü] → Ü, U ya da ü ...  re.IGNORECASE ile büyük/küçük farkı kalkar
KENDI_ADIM = re.compile(r"Sal[İIiı]h\s+Can\s+T[ÜUüu]rk", re.IGNORECASE)

# --- 3. Abonelik sahibi olarak geçen şirket adı ---
SIRKET = re.compile(r"OPTIMUM HAYAT SAGLIK[^,]*?STI\.*", re.IGNORECASE)

# --- 4. Hitap: artık boşluk, nokta ya da virgülden sonra da yakalar ---
# (?:...) = "yakalamayan grup": sadece gruplamak için, pandas uyarısını da önler
HITAP = re.compile(
    rf"\b(Sayın|Sayin|Sn|Sevgili|Merhaba)[\s.,]+(?!{GENEL})"
    r"([A-ZÇĞİÖŞÜ][a-zçğıöşü]+|[A-ZÇĞİÖŞÜ]{2,})"
    r"(?:\s+(?:[A-ZÇĞİÖŞÜ][a-zçğıöşü]+|[A-ZÇĞİÖŞÜ]{2,})){0,2}"
)

def ek_maskele(metin):
    metin = KENDI_ADIM.sub("[İSİM]", metin)      # önce kesin olanlar
    metin = SIRKET.sub("[İSİM]", metin)
    metin = UZUN_SAYI.sub("0000000", metin)
    metin = HITAP.sub(lambda m: f"{m.group(1)} [İSİM]", metin)   # sonra kalıplar
    return metin

ornek["mesaj"] = ornek["mesaj"].apply(ek_maskele)

# --- 5. Son kontrol: adın hâlâ bir yerde geçiyor mu? (0 olmalı) ---
kalan = ornek["mesaj"].str.contains(r"Sal[İIiı]h|T[ÜUüu]rk\b", case=False, regex=True)
print("Adın geçen satır:", kalan.sum())

# --- 6. Adres işaretli satırlar (yakalamayan grupla, uyarı çıkmaz) ---
ADRES = re.compile(r"\b(?:Mah\.?|Mahallesi|Sok\.?|Sokak|Cad\.?|Caddesi|No\s*:|Daire|Apt)\b")
print("Adres olabilecek satırlar:", [i + 2 for i in ornek.index[ornek["mesaj"].str.contains(ADRES)]])

# --- 7. Yaz ---
ornek.drop(columns=["mesaj_eski"]).to_csv(
    "../data/external/salih_kurumsal_ornek_maskeli.csv", index=False, encoding="utf-8-sig")

Adın geçen satır: 0
Adres olabilecek satırlar: [21, 35]


In [3]:
# --- Ek maskeleme v3: kaynak kodu, plaka, kart son 4, öndeki kodlar ---

# --- 1. v2'nin çıktısını oku (v2 doğruydu, üstüne ekliyoruz) ---
ornek = pd.read_csv("../data/external/salih_kurumsal_ornek_maskeli.csv")

# --- 2. Sondaki gönderici kodu: " B001", " B137" ... → sil ---
# $ = metnin sonu. Sadece en sondakini siler, metnin ortasındaki kodlara dokunmaz
ornek["mesaj"] = ornek["mesaj"].str.replace(r"\s*B\d{3}\s*$", "", regex=True)

# --- 3. Araç plakası: "06FV2316", "6 FV2316", "06 FV 2316" → [PLAKA] ---
PLAKA = re.compile(r"\b\d{1,2}\s?[A-Z]{1,3}\s?\d{2,4}\b")
ornek["mesaj"] = ornek["mesaj"].str.replace(PLAKA, "[PLAKA]", regex=True)

# --- 4. Kartın son 4 hanesi: "2016 ile biten kartiniz" → "0000 ile biten" ---
ornek["mesaj"] = ornek["mesaj"].str.replace(r"\b\d{4}(\s+ile biten)", r"0000\1", regex=True)

# --- 5. Önde duran kodlar: "856169 doğrulama", "G-293789 is your", "601-934." ---
ONDE_KOD = re.compile(
    r"\b(?:G-)?\d{3}-?\d{3}\b(?=\s*(?:doğrulama|dogrulama|is your|IsCep|\.?$))"
)
ornek["mesaj"] = ornek["mesaj"].str.replace(ONDE_KOD, "[KOD]", regex=True)
# "...kodunuz: 601-934." gibi öndeki kelimeden sonra gelenler
ornek["mesaj"] = ornek["mesaj"].str.replace(r"(kodunuz:?\s*)\d{3}-\d{3}", r"\1[KOD]", regex=True)
# "...code is: 433068." ve "...sifreniz:" gibi
ornek["mesaj"] = ornek["mesaj"].str.replace(r"(code is:?\s*)\d{6}", r"\1[KOD]", regex=True)

# --- 6. Kalan kısa abone/tekil numaralar: "87782 tekil numaraniz" ---
ornek["mesaj"] = ornek["mesaj"].str.replace(r"\b\d{5,6}(\s+tekil)", r"0000000\1", regex=True)

# --- 7. İngilizce mesajların dil etiketi ---
ingilizce = ornek["mesaj"].str.contains(r"\b(?:is your|verification code|Don't share)\b", regex=True)
ornek.loc[ingilizce, "dil"] = "en"

# --- 8. Kontroller ---
print("Sonda B kodu kalan:", ornek["mesaj"].str.contains(r"B\d{3}$").sum())          # 0 olmalı
print("Plaka kalan:", ornek["mesaj"].str.contains(r"\d{2}\s?FV").sum())               # 0 olmalı
print("6 haneli çıplak sayı kalan:", ornek["mesaj"].str.contains(r"\b\d{6}\b").sum())
print("dil=en:", ingilizce.sum())

# --- 9. Aynı dosyanın üstüne yaz (bu bizim ara çıktımız, orijinal değil) ---
ornek.to_csv("../data/external/salih_kurumsal_ornek_maskeli.csv", index=False, encoding="utf-8-sig")

Sonda B kodu kalan: 0
Plaka kalan: 0
6 haneli çıplak sayı kalan: 0
dil=en: 2


In [9]:
# --- Önerilen etiketleri uygula (Claude önerisi, Salih gözden geçirdi) ---

# --- 1. Dosyayı oku: bütün sütunlar metin, boş hücreler "" kalsın (NaN olmasın) ---
ornek = pd.read_csv("../data/external/salih_kurumsal_ornek_maskeli.csv",
                    dtype=str, keep_default_na=False)

# --- 2. Kaçan kod: satır 26 "63055 doğrulama kodunuz" ---
ornek["mesaj"] = ornek["mesaj"].str.replace(r"^\d{5}(\s+doğrulama)", r"[KOD]\1", regex=True)

# --- 3. Etiketler: anahtar = VS Code'daki satır numarası (1. satır başlık) ---
REKLAM = [5, 9, 10, 12, 14, 15, 18, 23, 28, 33, 36, 39, 44, 45, 47, 50, 56, 58,
          7, 19, 43, 60]                      # son dördü bahis
BAHIS = [7, 19, 43, 60]                       # Esra'nın kuralı: reklam + tur=bahis
BELIRSIZ = [46, 59]                           # OEDAS uygulama tanıtımı: meşru mu reklam mı?

satir_no = ornek.index + 2                    # pandas 0'dan, VS Code 2'den başlıyor
ornek["etiket"] = "meşru"                     # varsayılan: meşru
ornek.loc[satir_no.isin(REKLAM), "etiket"] = "reklam"
ornek.loc[satir_no.isin(BAHIS), "tur"] = "bahis"
ornek.loc[satir_no.isin(BELIRSIZ), "belirsiz"] = "e"
ornek["etiketleyen"] = "Salih"

# --- 4. Özet ---
print(ornek["etiket"].value_counts().to_dict(), "| belirsiz:", (ornek["belirsiz"] == "e").sum())
print("satır 26:", ornek.loc[24, "mesaj"][:40])

# --- 5. Son dosya (data/external + "_etiketli" → iki ayrı kuralla gitignore'da) ---
ornek.to_csv("../data/external/salih_kurumsal_etiketli.csv", index=False, encoding="utf-8-sig")

{'meşru': 38, 'reklam': 22} | belirsiz: 2
satır 26: [KOD] doğrulama kodunuz ile seçtiğiniz f
